# ARC_ATLAS v4 Small-Lesion Runbook

Current experiment:
- center-crop full MNI brain to `192 x 224 x 192`
- extract 3 deterministic overlapping windows along canonical left/right axis `x`
- windows are:
  - left: `0:96`
  - center: `48:144`
  - right: `96:192`
- train dense segmentation only for now; proposal/aux heads remain off until the window geometry is validated

Use the preview widget below before starting a run.


## Three-Window Preview

Run the next cell before training. It uses the exact current training geometry:
- canonical MNI volume
- center-crop to `192 x 224 x 192`
- exact overlapping windows along canonical `x`:
  - left: `0:96`
  - center: `48:144`
  - right: `96:192`

The widget lets you choose the dataset and MRI case. It overlays the lesion mask and shows the full cropped brain plus all three training windows in axial and coronal views.


In [1]:
# --------- Three-window preview widget (run before training) ---------
from pathlib import Path
import importlib.util
from collections import defaultdict

import csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SRC_PREVIEW = PROJECT_ROOT / "src" / "training_v2_smalllesion.py"
TRAIN_DIR_PREVIEW = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train")
PREVIEW_MANIFEST = TRAIN_DIR_PREVIEW / "manifest.csv"

PREVIEW_CROP_SHAPE = (192, 224, 192)
PREVIEW_AXIS = 0
WINDOWS = [
    ("Left", 0, 96),
    ("Center", 48, 144),
    ("Right", 96, 192),
]

if not SRC_PREVIEW.exists():
    raise FileNotFoundError(f"Training module not found: {SRC_PREVIEW}")
if not PREVIEW_MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {PREVIEW_MANIFEST}")

spec_preview = importlib.util.spec_from_file_location("seg_preview", SRC_PREVIEW)
if spec_preview is None or spec_preview.loader is None:
    raise RuntimeError(f"Could not load module spec from {SRC_PREVIEW}")
seg_preview = importlib.util.module_from_spec(spec_preview)
spec_preview.loader.exec_module(seg_preview)


def _pair_source(img_path: Path) -> str:
    name = img_path.name
    return name.split("__", 1)[0] if "__" in name else name.split("_", 1)[0]


def _load_pairs_from_manifest(manifest_path: Path):
    pairs = []
    with manifest_path.open(newline="") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            img_p = Path(row["t1"])
            msk_p = Path(row["mask"])
            if img_p.exists() and msk_p.exists():
                pairs.append((img_p, msk_p))
    if not pairs:
        raise RuntimeError(f"No valid image/mask pairs found in {manifest_path}")
    return pairs


def _load_case(img_path: Path, msk_path: Path, crop_shape):
    img = seg_preview._load_vol_canonical(str(img_path))
    msk = (seg_preview._load_vol_canonical(str(msk_path)) > 0.5).astype(np.float32)
    img_c = seg_preview._center_crop_or_pad_volume(img, crop_shape)
    msk_c = (seg_preview._center_crop_or_pad_volume(msk, crop_shape) > 0.5).astype(np.float32)
    return img, img_c, msk_c


def _lesion_center(mask):
    coords = np.argwhere(mask > 0.5)
    if coords.size == 0:
        x, y, z = mask.shape
        return x // 2, y // 2, z // 2
    return tuple(np.round(coords.mean(axis=0)).astype(int))


def _contour(ax, arr2d, color, linewidth=1.25):
    if np.any(arr2d > 0):
        ax.contour(arr2d.astype(np.float32), levels=[0.5], colors=[color], linewidths=linewidth)


def _show_case(img_native, img_crop, mask_crop, title):
    xc, yc, zc = _lesion_center(mask_crop)

    axial_img = img_crop[:, :, zc].T
    axial_msk = mask_crop[:, :, zc].T
    cor_img = img_crop[:, yc, :].T
    cor_msk = mask_crop[:, yc, :].T

    fig, axes = plt.subplots(2, 4, figsize=(17, 8))
    counts = []
    for _, lo, hi in WINDOWS:
        counts.append(int(mask_crop[lo:hi, :, :].sum()))
    fig.suptitle(
        f"{title}\n"
        f"native={img_native.shape} cropped={img_crop.shape} | lesion voxels left/center/right={counts[0]}/{counts[1]}/{counts[2]}",
        fontsize=11,
    )

    axes[0, 0].imshow(axial_img, cmap="gray", origin="lower")
    _contour(axes[0, 0], axial_msk, "lime")
    for split in [48 - 0.5, 96 - 0.5, 144 - 0.5]:
        axes[0, 0].axvline(split, color="cyan", linestyle="--", linewidth=1.1)
    axes[0, 0].set_title(f"Full axial z={zc}")

    axes[1, 0].imshow(cor_img, cmap="gray", origin="lower")
    _contour(axes[1, 0], cor_msk, "lime")
    for split in [48 - 0.5, 96 - 0.5, 144 - 0.5]:
        axes[1, 0].axvline(split, color="cyan", linestyle="--", linewidth=1.1)
    axes[1, 0].set_title(f"Full coronal y={yc}")

    for col, (label, lo, hi) in enumerate(WINDOWS, start=1):
        patch_img = img_crop[lo:hi, :, :]
        patch_msk = mask_crop[lo:hi, :, :]
        axial_patch = patch_img[:, :, zc].T
        axial_patch_msk = patch_msk[:, :, zc].T
        cor_patch = patch_img[:, yc, :].T
        cor_patch_msk = patch_msk[:, yc, :].T

        axes[0, col].imshow(axial_patch, cmap="gray", origin="lower")
        _contour(axes[0, col], axial_patch_msk, "lime")
        axes[0, col].set_title(f"{label} axial x={lo}:{hi}")

        axes[1, col].imshow(cor_patch, cmap="gray", origin="lower")
        _contour(axes[1, col], cor_patch_msk, "lime")
        axes[1, col].set_title(f"{label} coronal x={lo}:{hi}")

    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()


pairs_preview = _load_pairs_from_manifest(PREVIEW_MANIFEST)
by_source = defaultdict(list)
for img_p, msk_p in pairs_preview:
    by_source[_pair_source(img_p)].append((img_p, msk_p))
for source in by_source:
    by_source[source] = sorted(by_source[source], key=lambda t: t[0].name)

source_options = sorted(by_source.keys())
source_dd = widgets.Dropdown(options=source_options, description='Dataset:', layout=widgets.Layout(width='450px'))
case_dd = widgets.Dropdown(description='MRI:', layout=widgets.Layout(width='900px'))
refresh_btn = widgets.Button(description='Refresh View', button_style='info')
out = widgets.Output()


def _case_options(source_name):
    return [(img_p.name, img_p.name) for img_p, _ in by_source[source_name]]


def _update_case_options(*args):
    case_dd.options = _case_options(source_dd.value)
    if case_dd.options:
        case_dd.value = case_dd.options[0][1]


def _render(*args):
    with out:
        clear_output(wait=True)
        source_name = source_dd.value
        case_name = case_dd.value
        pair = next((pair for pair in by_source[source_name] if pair[0].name == case_name), None)
        if pair is None:
            print('No matching case selected.')
            return
        img_p, msk_p = pair
        print('Exact windows along canonical x: left 0:96 | center 48:144 | right 96:192')
        img_native, img_crop, mask_crop = _load_case(img_p, msk_p, PREVIEW_CROP_SHAPE)
        _show_case(img_native, img_crop, mask_crop, title=f"{source_name} | {img_p.name}")


source_dd.observe(_update_case_options, names='value')
case_dd.observe(_render, names='value')
refresh_btn.on_click(_render)
_update_case_options()
controls = widgets.VBox([
    widgets.HTML('<b>Browse cropped three-window splits</b>'),
    source_dd,
    case_dd,
    refresh_btn,
])
display(controls, out)
_render()


2026-04-14 18:00:29.382220: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1776211231.201116 3201344 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1776211231.202151 3201344 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5443 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1776211231.202497 3201344 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1776211231.203573 3201344 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13348 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2026-04-14 18:00:31,272 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-04-14 18:00:31,273 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-04-14 18:00:31,274 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow

Strategy: MirroredStrategy


Output()

In [ ]:
from pathlib import Path
import importlib.util
import shutil
import time
import traceback

# --------- Paths and module loading ---------
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
RUN_ROOT = PROJECT_ROOT
SRC = PROJECT_ROOT / "src" / "training_v2_smalllesion.py"

TRAIN_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train")
TRAIN_T1 = TRAIN_DIR / "t1"
TRAIN_MASKS = TRAIN_DIR / "masks"

if not SRC.exists():
    raise FileNotFoundError(f"Training module not found: {SRC}")
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"Training data dir not found: {TRAIN_DIR}. Run ARC_ATLAS_TrainPrep_v4.ipynb first.")
if not TRAIN_T1.exists() or not TRAIN_MASKS.exists():
    raise FileNotFoundError(f"Expected subfolders missing under {TRAIN_DIR}: t1/ and masks/")

spec = importlib.util.spec_from_file_location("seg", SRC)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load module spec from {SRC}")
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# --------- Hyperparameters (v3-like whole-brain core on the v4 trainer) ---------
# NOTE: the preview cell above still visualizes three-window geometry, but training below now uses full-brain patches.
INPUT_SHAPE = (192, 224, 192, 1)
PATCH_SIZE = (192, 224, 192)
PATCHES_PER_CASE = 1
EPOCH_STEPS = 258  # match the effective v3 whole-brain steps/epoch while keeping v4 sampling manageable
FIT_VERBOSE = 2
MEMORY_LOGS_ENABLED = False
DIAGNOSTICS_ENABLED = True
BATCH_LOG_EVERY_N_STEPS = 1
DIAGNOSTICS_COMPARE_POSTPROC = True
TOTAL_EPOCHS = 140
INITIAL_EPOCH = 0

BASE_FILTERS = 8
SAM_HEADS = 2
BATCH_SIZE = 1
VAL_SPLIT = 0.15
DROPOUT_RATE = 0.55
L2_REG = 1.5e-3

AUG_INTENSITY = 0.30
ROTATION_RANGE = 20
SMALL_LESION_THRESHOLD = 100
SYNTHETIC_LESION_PROB = 0.3

INITIAL_LR = 1e-4
MIN_LR = 5e-7
WARMUP_EPOCHS = 15
COSINE_FIRST_CYCLE_EPOCHS = 70
COSINE_T_MUL = 1.0
COSINE_M_MUL = 1.0
SWA_EPOCHS = 0
SWA_LR_MULT = None

DICE_WEIGHT = 0.40
BOUNDARY_WEIGHT = 0.60
BCE_WEIGHT = 0.20
VOLUME_RATIO_WEIGHT = 0.05
BOUNDARY_WARMUP_DICE = 0.4
BOUNDARY_WARMUP_BOUNDARY = 0.6
BOUNDARY_RAMP_EPOCHS = 1

FOCAL_TVERSKY_WEIGHT = 0.0
TVERSKY_ALPHA = 0.7
TVERSKY_BETA = 0.3
FOCAL_TVERSKY_GAMMA = 1.5

SIZE_BUCKET_PROBS = (0.45, 0.25, 0.15, 0.10, 0.05)
PATCH_FG_PROB_BY_BIN = (0.995, 0.98, 0.90, 0.75)
SOURCE_BALANCED_SAMPLING = True
OUTPUT_BIAS_INIT_PROB = 0.015
USE_SYMMETRIC_FLIP_CHANNEL = False
CASE_SIZE_BINS = (100, 1000, 10000)
CASE_SIZE_GROUP_PROBS = (0.45, 0.25, 0.15, 0.10)
CASE_NONE_PROB = 0.05
USE_COMPONENT_AWARE_PATCH_SAMPLING = False
USE_TINY_COMPONENT_CENTERING = False
TINY_COMPONENT_CENTER_PROB = 0.95
SMALL_COMPONENT_CENTER_PROB = 0.85
TINY_COMPONENT_MAX_JITTER = 2
SMALL_COMPONENT_MAX_JITTER = 4
MSL_COMPONENT_THRESHOLDS = (100, 1000, 10000)
USE_CENTER_HEATMAP_HEAD = False
USE_SIZE_HEAD = False
CENTER_HEATMAP_SIGMA = 4.0
CENTER_POSITIVE_WEIGHT = 10.0
AUX_CENTER_WEIGHT = 0.12
AUX_SIZE_WEIGHT = 0.05
SIZE_HEAD_CLASS_WEIGHTS = (0.02, 4.0, 2.5, 1.0, 0.6)
CENTER_TOPK_VALUES = (1, 3, 5, 10, 20)
CENTER_MATCH_RADIUS = 6.0
CENTER_NMS_RADIUS = 6
CENTER_MIN_CONFIDENCE = 0.01
CENTER_HEAD_BIAS_INIT_PROB = 0.01
CENTER_LOSS_GAMMA = 2.0
CENTER_LOSS_BETA = 4.0
USE_AUX_MSL_HEAD = False
USE_AUX_DBL_HEAD = False
AUX_MSL_WEIGHT = 0.00
AUX_DBL_WEIGHT = 0.00
AUX_MSL_CLASS_WEIGHTS = (0.02, 4.0, 2.5, 1.0, 0.6)
AUX_DBL_CLASS_WEIGHTS = (0.02, 1.15, 1.0)
USE_SIZE_CURRICULUM = False
CURRICULUM_EPOCHS = 12
CURRICULUM_START_CASE_GROUP_PROBS = (0.70, 0.20, 0.07, 0.03)
CURRICULUM_START_PATCH_FG_PROB_BY_BIN = (0.995, 0.99, 0.92, 0.78)
CURRICULUM_START_CASE_NONE_PROB = 0.02
USE_ATLAS_FINE_TUNE = False
ATLAS_FINE_TUNE_START_EPOCH = 70
ATLAS_FINE_TUNE_SOURCE_PREFIXES = ("ATLAS",)
ATLAS_FINE_TUNE_SOURCE_MASS = 0.70
ATLAS_FINE_TUNE_CASE_GROUP_PROBS = (0.60, 0.25, 0.10, 0.05)
ATLAS_FINE_TUNE_PATCH_FG_PROB_BY_BIN = (0.995, 0.985, 0.94, 0.82)
ATLAS_FINE_TUNE_CASE_NONE_PROB = 0.02
TOPK_VOXEL_FRACTION = 0.00
TOPK_WEIGHT = 0.00
LESION_INSERTION_PROB = 0.00
LESION_INSERTION_MAX_COMPONENT_VOXELS = 1000
USE_BRAINMASK_POSTPROC = False
USE_COMPONENT_SCORING_POSTPROC = False
GROUPED_CV_FOLDS = 3
EXTERNAL_VAL_DIR = None
EXTERNAL_VAL_MANIFEST = None

# Full-image patch extraction controls
LOAD_FULL_IMAGE_FOR_PATCHING = True
FULL_RES_TARGET_SHAPE = (192, 224, 192)
WHOLE_BRAIN_VAL_ENABLED = True
WHOLE_BRAIN_VAL_EVERY_N_EPOCHS = 1
WHOLE_BRAIN_VAL_MAX_CASES = None
WHOLE_BRAIN_VAL_TTA = False
PATCH_SAMPLING_STRATEGY = "random"
HEMISPHERE_AXIS = 0
HEMISPHERE_BALANCED = True

# --------- Per-run artifact directories ---------
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
for d in (MODEL_DIR, CALLBACKS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Using training module:", SRC)
print("Training data:", TRAIN_DIR)
print("Run dir:", RUN_DIR)
ACTIVE_SMALL_LESION_MEASURES = {
    "source_balanced_sampling": SOURCE_BALANCED_SAMPLING,
    "case_size_bins": CASE_SIZE_BINS,
    "case_size_group_probs": CASE_SIZE_GROUP_PROBS,
    "case_none_prob": CASE_NONE_PROB,
    "patch_fg_prob_by_bin": PATCH_FG_PROB_BY_BIN,
    "component_aware_patch_sampling": USE_COMPONENT_AWARE_PATCH_SAMPLING,
    "tiny_component_centering": USE_TINY_COMPONENT_CENTERING,
    "center_heatmap_head": USE_CENTER_HEATMAP_HEAD,
    "size_head": USE_SIZE_HEAD,
    "center_heatmap_sigma": CENTER_HEATMAP_SIGMA,
    "center_head_bias_init_prob": CENTER_HEAD_BIAS_INIT_PROB,
    "center_loss_gamma": CENTER_LOSS_GAMMA,
    "center_loss_beta": CENTER_LOSS_BETA,
    "center_topk_values": CENTER_TOPK_VALUES,
    "center_match_radius": CENTER_MATCH_RADIUS,
    "size_head_class_weights": SIZE_HEAD_CLASS_WEIGHTS,
    "aux_msl_class_weights": AUX_MSL_CLASS_WEIGHTS,
    "aux_dbl_class_weights": AUX_DBL_CLASS_WEIGHTS,
    "size_curriculum": USE_SIZE_CURRICULUM,
    "curriculum_epochs": CURRICULUM_EPOCHS,
    "curriculum_start_case_group_probs": CURRICULUM_START_CASE_GROUP_PROBS,
    "curriculum_start_patch_fg_prob_by_bin": CURRICULUM_START_PATCH_FG_PROB_BY_BIN,
    "atlas_fine_tune": USE_ATLAS_FINE_TUNE,
    "atlas_fine_tune_start_epoch": ATLAS_FINE_TUNE_START_EPOCH,
    "atlas_fine_tune_source_mass": ATLAS_FINE_TUNE_SOURCE_MASS,
    "brainmask_postproc": USE_BRAINMASK_POSTPROC,
    "component_scoring_postproc": USE_COMPONENT_SCORING_POSTPROC,
    "aux_msl_head": USE_AUX_MSL_HEAD,
    "aux_dbl_head": USE_AUX_DBL_HEAD,
    "topk_weight": TOPK_WEIGHT,
    "lesion_insertion_prob": LESION_INSERTION_PROB,
    "symmetric_flip_channel": USE_SYMMETRIC_FLIP_CHANNEL,
    "initial_lr": INITIAL_LR,
    "total_epochs": TOTAL_EPOCHS,
    "dropout_rate": DROPOUT_RATE,
}
print("Active v4 robustness measures:")
for k, v in ACTIVE_SMALL_LESION_MEASURES.items():
    print(f" - {k}: {v}")

# --------- Train whole-brain v4 run (v3-like core settings, v4 machinery retained) ---------
try:
    history = seg.train_dynamic_model(
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,
        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,
        INPUT_SHAPE=INPUT_SHAPE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        BATCH_SIZE=BATCH_SIZE,
        DROPOUT_RATE=DROPOUT_RATE,
        L2_REG=L2_REG,
        PATCH_SIZE=PATCH_SIZE,
        PATCHES_PER_CASE=PATCHES_PER_CASE,
        EPOCH_STEPS=EPOCH_STEPS,
        FIT_VERBOSE=FIT_VERBOSE,
        MEMORY_LOGS_ENABLED=MEMORY_LOGS_ENABLED,
        DIAGNOSTICS_ENABLED=DIAGNOSTICS_ENABLED,
        BATCH_LOG_EVERY_N_STEPS=BATCH_LOG_EVERY_N_STEPS,
        DIAGNOSTICS_COMPARE_POSTPROC=DIAGNOSTICS_COMPARE_POSTPROC,
        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        RESAMPLE_TO_TARGET=True,
        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        ROTATION_RANGE=ROTATION_RANGE,
        SMALL_LESION_THRESHOLD=SMALL_LESION_THRESHOLD,
        SYNTHETIC_LESION_PROB=SYNTHETIC_LESION_PROB,
        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,
        COSINE_FIRST_CYCLE_EPOCHS=COSINE_FIRST_CYCLE_EPOCHS,
        COSINE_T_MUL=COSINE_T_MUL,
        COSINE_M_MUL=COSINE_M_MUL,
        COSINE_MIN_LR_MULT=0.1,
        SWA_EPOCHS=SWA_EPOCHS,
        SWA_LR_MULT=SWA_LR_MULT,
        DICE_WEIGHT=DICE_WEIGHT,
        BOUNDARY_WEIGHT=BOUNDARY_WEIGHT,
        BCE_WEIGHT=BCE_WEIGHT,
        VOLUME_RATIO_WEIGHT=VOLUME_RATIO_WEIGHT,
        DICE_LOSS_WEIGHT=0.4,
        BOUNDARY_LOSS_WEIGHT=0.6,
        BOUNDARY_WARMUP_DICE=BOUNDARY_WARMUP_DICE,
        BOUNDARY_WARMUP_BOUNDARY=BOUNDARY_WARMUP_BOUNDARY,
        BOUNDARY_RAMP_EPOCHS=BOUNDARY_RAMP_EPOCHS,
        FOCAL_TVERSKY_WEIGHT=FOCAL_TVERSKY_WEIGHT,
        TVERSKY_ALPHA=TVERSKY_ALPHA,
        TVERSKY_BETA=TVERSKY_BETA,
        FOCAL_TVERSKY_GAMMA=FOCAL_TVERSKY_GAMMA,
        SIZE_BUCKET_PROBS=SIZE_BUCKET_PROBS,
        PATCH_FG_PROB_BY_BIN=PATCH_FG_PROB_BY_BIN,
        SOURCE_BALANCED_SAMPLING=SOURCE_BALANCED_SAMPLING,
        OUTPUT_BIAS_INIT_PROB=OUTPUT_BIAS_INIT_PROB,
        USE_SYMMETRIC_FLIP_CHANNEL=USE_SYMMETRIC_FLIP_CHANNEL,
        CASE_SIZE_BINS=CASE_SIZE_BINS,
        CASE_SIZE_GROUP_PROBS=CASE_SIZE_GROUP_PROBS,
        CASE_NONE_PROB=CASE_NONE_PROB,
        USE_COMPONENT_AWARE_PATCH_SAMPLING=USE_COMPONENT_AWARE_PATCH_SAMPLING,
        USE_TINY_COMPONENT_CENTERING=USE_TINY_COMPONENT_CENTERING,
        TINY_COMPONENT_CENTER_PROB=TINY_COMPONENT_CENTER_PROB,
        SMALL_COMPONENT_CENTER_PROB=SMALL_COMPONENT_CENTER_PROB,
        TINY_COMPONENT_MAX_JITTER=TINY_COMPONENT_MAX_JITTER,
        SMALL_COMPONENT_MAX_JITTER=SMALL_COMPONENT_MAX_JITTER,
        MSL_COMPONENT_THRESHOLDS=MSL_COMPONENT_THRESHOLDS,
        USE_CENTER_HEATMAP_HEAD=USE_CENTER_HEATMAP_HEAD,
        USE_SIZE_HEAD=USE_SIZE_HEAD,
        CENTER_HEATMAP_SIGMA=CENTER_HEATMAP_SIGMA,
        CENTER_POSITIVE_WEIGHT=CENTER_POSITIVE_WEIGHT,
        AUX_CENTER_WEIGHT=AUX_CENTER_WEIGHT,
        AUX_SIZE_WEIGHT=AUX_SIZE_WEIGHT,
        SIZE_HEAD_CLASS_WEIGHTS=SIZE_HEAD_CLASS_WEIGHTS,
        CENTER_TOPK_VALUES=CENTER_TOPK_VALUES,
        CENTER_MATCH_RADIUS=CENTER_MATCH_RADIUS,
        CENTER_NMS_RADIUS=CENTER_NMS_RADIUS,
        CENTER_MIN_CONFIDENCE=CENTER_MIN_CONFIDENCE,
        CENTER_HEAD_BIAS_INIT_PROB=CENTER_HEAD_BIAS_INIT_PROB,
        CENTER_LOSS_GAMMA=CENTER_LOSS_GAMMA,
        CENTER_LOSS_BETA=CENTER_LOSS_BETA,
        USE_AUX_MSL_HEAD=USE_AUX_MSL_HEAD,
        USE_AUX_DBL_HEAD=USE_AUX_DBL_HEAD,
        AUX_MSL_WEIGHT=AUX_MSL_WEIGHT,
        AUX_DBL_WEIGHT=AUX_DBL_WEIGHT,
        AUX_MSL_CLASS_WEIGHTS=AUX_MSL_CLASS_WEIGHTS,
        AUX_DBL_CLASS_WEIGHTS=AUX_DBL_CLASS_WEIGHTS,
        USE_SIZE_CURRICULUM=USE_SIZE_CURRICULUM,
        CURRICULUM_EPOCHS=CURRICULUM_EPOCHS,
        CURRICULUM_START_CASE_GROUP_PROBS=CURRICULUM_START_CASE_GROUP_PROBS,
        CURRICULUM_START_PATCH_FG_PROB_BY_BIN=CURRICULUM_START_PATCH_FG_PROB_BY_BIN,
        CURRICULUM_START_CASE_NONE_PROB=CURRICULUM_START_CASE_NONE_PROB,
        USE_ATLAS_FINE_TUNE=USE_ATLAS_FINE_TUNE,
        ATLAS_FINE_TUNE_START_EPOCH=ATLAS_FINE_TUNE_START_EPOCH,
        ATLAS_FINE_TUNE_SOURCE_PREFIXES=ATLAS_FINE_TUNE_SOURCE_PREFIXES,
        ATLAS_FINE_TUNE_SOURCE_MASS=ATLAS_FINE_TUNE_SOURCE_MASS,
        ATLAS_FINE_TUNE_CASE_GROUP_PROBS=ATLAS_FINE_TUNE_CASE_GROUP_PROBS,
        ATLAS_FINE_TUNE_PATCH_FG_PROB_BY_BIN=ATLAS_FINE_TUNE_PATCH_FG_PROB_BY_BIN,
        ATLAS_FINE_TUNE_CASE_NONE_PROB=ATLAS_FINE_TUNE_CASE_NONE_PROB,
        TOPK_VOXEL_FRACTION=TOPK_VOXEL_FRACTION,
        TOPK_WEIGHT=TOPK_WEIGHT,
        LESION_INSERTION_PROB=LESION_INSERTION_PROB,
        LESION_INSERTION_MAX_COMPONENT_VOXELS=LESION_INSERTION_MAX_COMPONENT_VOXELS,
        USE_BRAINMASK_POSTPROC=USE_BRAINMASK_POSTPROC,
        USE_COMPONENT_SCORING_POSTPROC=USE_COMPONENT_SCORING_POSTPROC,
        GROUPED_CV_FOLDS=GROUPED_CV_FOLDS,
        EXTERNAL_VAL_DIR=EXTERNAL_VAL_DIR,
        EXTERNAL_VAL_MANIFEST=EXTERNAL_VAL_MANIFEST,
        LOAD_FULL_IMAGE_FOR_PATCHING=LOAD_FULL_IMAGE_FOR_PATCHING,
        FULL_RES_TARGET_SHAPE=FULL_RES_TARGET_SHAPE,
        WHOLE_BRAIN_VAL_ENABLED=WHOLE_BRAIN_VAL_ENABLED,
        WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=WHOLE_BRAIN_VAL_EVERY_N_EPOCHS,
        WHOLE_BRAIN_VAL_MAX_CASES=WHOLE_BRAIN_VAL_MAX_CASES,
        WHOLE_BRAIN_VAL_TTA=WHOLE_BRAIN_VAL_TTA,
        PATCH_SAMPLING_STRATEGY=PATCH_SAMPLING_STRATEGY,
        HEMISPHERE_AXIS=HEMISPHERE_AXIS,
        HEMISPHERE_BALANCED=HEMISPHERE_BALANCED,
        DIFF_AWARE_ENABLED=True,
        DIFF_EMA_LAMBDA=0.8,
        DIFF_BETA=1.5,
        VALIDATION_SPLIT=VAL_SPLIT,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,
    )
    print("Training complete. Keys:", list(getattr(history, "history", {}).keys()))
    print("Artifacts saved to", RUN_DIR)
except Exception:
    traceback.print_exc()
    raise

# Convenience: mark this run as latest
latest_link = RUN_ROOT / "runs" / "latest"
if latest_link.exists() or latest_link.is_symlink():
    latest_link.unlink()
latest_link.symlink_to(RUN_DIR, target_is_directory=True)

best_src = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if best_src.exists():
    best_copy = RUN_ROOT / "runs" / "latest_best.weights.h5"
    shutil.copy2(best_src, best_copy)
    print("Saved best copy ->", best_copy)





Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2026-04-14 18:00:33,598 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2026-04-14 18:00:33,602 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-04-14 18:00:33,602 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-04-14 18:00:33,603 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.6
- GPU devices: 2
2026-04-14 18:00:33,615 - SmartSOTA_Dynamic - INFO - 🔧 Config: smart_sota_dynamic_20260414_180033.keras
2026-04-14 18:00:33,615 - SmartSOTA_Dynamic - INFO - Small-lesion trainer enabled: flip_channel=False center_head=False size_head=False aux_msl=False aux_dbl=False topk=0.00 lesion_insertion=0.00 component_patch_sampling=False tiny_centering=False brainmask_postproc=False component_postproc=False case_group_probs=(0.45, 0.25, 0.15, 0.1) case_none_prob=0.050 patch_f

Strategy: MirroredStrategy
Using training module: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/src/training_v2_smalllesion.py
Training data: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train
Run dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260414_180033
Active v4 robustness measures:
 - source_balanced_sampling: True
 - case_size_bins: (100, 1000, 10000)
 - case_size_group_probs: (0.45, 0.25, 0.15, 0.1)
 - case_none_prob: 0.05
 - patch_fg_prob_by_bin: (0.995, 0.98, 0.9, 0.75)
 - component_aware_patch_sampling: False
 - tiny_component_centering: False
 - center_heatmap_head: False
 - size_head: False
 - center_heatmap_sigma: 4.0
 - center_head_bias_init_prob: 0.01
 - center_loss_gamma: 2.0
 - center_loss_beta: 4.0
 - center_topk_values: (1, 3, 5, 10, 20)
 - center_match_radius: 6.0
 - size_head_class_weights: (0.02, 4.0, 2.5, 1.0, 0.6)
 - aux_msl_class_weights: (0.02, 4.0, 2.5, 1.0

2026-04-14 18:00:34,819 - SmartSOTA_Dynamic - INFO - Model built: 2,786,729 parameters
2026-04-14 18:00:34,820 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2026-04-14 18:00:34,820 - SmartSOTA_Dynamic - INFO - 📄 Using manifest-defined pairs from /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
2026-04-14 18:02:19,140 - SmartSOTA_Dynamic - INFO - Manifest composition: {'ARC-combined-t1-raw-ab0d1794': 190, 'ATLAS-Images-f0d7431e': 582, 'Approx-Numeracy-Processed': 94}
2026-04-14 18:02:19,141 - SmartSOTA_Dynamic - INFO - 📊 Created 866 image–mask pairs from manifest
2026-04-14 18:02:19,142 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 99.65%
2026-04-14 18:10:17,928 - SmartSOTA_Dynamic - INFO - 🧮 Dataset split (stratified_source): Train=736 (85.0%), Validation=130 (15.0%)
2026-04-14 18:10:17,929 - SmartSOTA_Dynamic - INFO - 🧩 Stratification groups: {'ARC-combined-t1-raw-ab0d1794': 190, 'ATLAS-Im

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


...

2026-04-14 18:10:29,233 - tensorflow - INFO - Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
2026-04-14 18:10:29,236 - SmartSOTA_Dynamic - INFO - Sampling policy @epoch 0: phase=steady case_group_probs=(0.45, 0.25, 0.15, 0.1) patch_fg_probs=(0.995, 0.98, 0.9, 0.75) case_none_prob=0.050 source_overrides={}
2026-04-14 18:10:29,237 - SmartSOTA_Dynamic - INFO - 📉 Loss mix @epoch 0: dice=0.400, boundary=0.600, bce=0.200, topk=0.000, volume=0.050, focal=0.000


Epoch 1/140
INFO:tensorflow:Collective all_reduce tensors: 167 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


2026-04-14 18:10:33,021 - tensorflow - INFO - Collective all_reduce tensors: 167 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
2026-04-14 18:10:51.190122: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001
2026-04-14 18:10:51.205329: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001
2026-04-14 18:10:53.540776: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:361] gpu_async_0 cuMemAllocAsync failed to allocate 2659732656 bytes: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
 Reported by CUDA: Free memory/Total memory: 1595342848/25261047808
2026-04-14 18:10:53.540806: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:366] Stats: Limit:                      5707857920
InUse:                      4508112600
MaxInUse:                   6441653428
NumAllocs:                     

ResourceExhaustedError: Graph execution error:

Detected at node replica_1/SmartSOTA_SmallLesion_1/vision_mamba_block_7_1/layer_normalization_34_1/mul_2 defined at (most recent call last):
  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/threading.py", line 973, in _bootstrap

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/threading.py", line 1016, in _bootstrap_inner

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/keras/src/backend/tensorflow/trainer.py", line 58, in train_step

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 117, in error_handler

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/keras/src/layers/layer.py", line 941, in __call__

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 117, in error_handler

  File "/home/rbielski/miniconda3/envs/tf_310/lib/pytho

: 

In [ ]:
# --------- Quick sanity prediction on zeros ---------
import numpy as np

cfg = seg.DynamicTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_T1,
    MASKS_DIR=TRAIN_MASKS,
    INPUT_SHAPE=INPUT_SHAPE,
    PATCH_SIZE=PATCH_SIZE,
    MODEL_DIR=MODEL_DIR,
    CALLBACKS_DIR=CALLBACKS_DIR,
)

weights = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if weights.exists():
    m = seg.build_model_for_inference(cfg, weights_path=str(weights))
else:
    m = seg.build_model_for_inference(cfg)

x0 = np.zeros((1, *INPUT_SHAPE), np.float32)
p0 = m.predict(x0, verbose=0)[0, ..., 0]
print("Blank input -> p.mean=", float(p0.mean()), " p.max=", float(p0.max()))


Blank input -> p.mean= 0.10394287109375  p.max= 0.95703125
